# Epsilon Fund — Master Portfolio

Combined OOS portfolio spanning **every strategy bucket** in the repo.

- **Stat Arb**  — `topics/statistical-arbitrage/strategies/testing/*_oos.pkl`  (baked `net_returns`)
- **Momentum** — `topics/momentum/strategies/<strategy>/oos/*_oos.pkl`  (raw `position` / `Close`)

Reuses the `portfolio_master` template end-to-end: every sleeve is wrapped into a unified schema so `plot_portfolio_oos` handles the main chart, metrics, yearly tables, cost stress — exactly as it does for the momentum-only notebook. Added on top: the bucket comparison, correlation, and weight-sweep sensitivity from `combined_portfolio.py`.

Individual signals are never modified — behaviour is entirely driven by each sleeve's `*_oos.pkl`.

Scaling: drop a new folder into `MOMENTUM_STRATEGIES`, add entries to `MOMENTUM_SELECTION`, rerun.

---

In [ ]:
import os, sys, io, contextlib
import pandas as pd

# ── repo root — auto-detected from known directory markers ───────────────────
def _find_root():
    cand = os.path.abspath('')
    for _ in range(6):
        if os.path.isdir(os.path.join(cand, 'infrastructure')) and \
           os.path.isdir(os.path.join(cand, 'topics')):
            return cand
        cand = os.path.dirname(cand)
    raise RuntimeError('Cannot find repo root — launch Jupyter from inside the repo')

ROOT = _find_root()

STATARB_DIR = os.path.join(ROOT, 'topics', 'statistical-arbitrage', 'strategies', 'testing')
MOM_ROOT    = os.path.join(ROOT, 'topics', 'momentum', 'strategies')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))
from wf_visualizer import plot_portfolio_oos, plot_closed_trade_equity
from engine import backtest
from portfolio_metrics import (
    load_pkl,
    mom_bar_returns, wrap_as_sleeve, sleeve_freq,
    norm_weights, sa_inverse_vol_weights,
    build_momentum_weights, build_sleeve_weights,
    sweep_top_level, sweep_momentum_strategy,
)

---
## 1 · Strategy Registries

Declare every momentum strategy folder you want available — stat arb is discovered automatically from the flat testing directory. Everything downstream keys off these registries.

In [ ]:
# ── momentum strategies: tag → folder under topics/momentum/strategies/ ──────
MOMENTUM_STRATEGIES = {
    'wf2': 'wf_testing_2',
    'bb':  'bb_breakout_wf',
    # 'xxx': 'new_wf_folder',   # ← drop new momentum WF strategies in here
}

# ── load stat arb OOS ────────────────────────────────────────────────────────
statarb_oos = {}
for fname in sorted(os.listdir(STATARB_DIR)):
    if fname.endswith('_oos.pkl'):
        pair = fname.replace('_oos.pkl', '').upper()
        statarb_oos[pair] = load_pkl(os.path.join(STATARB_DIR, fname))

# ── load momentum OOS ────────────────────────────────────────────────────────
momentum_oos = {}
for tag, folder in MOMENTUM_STRATEGIES.items():
    path = os.path.join(MOM_ROOT, folder, 'oos')
    if not os.path.isdir(path):
        print(f'  [skip] momentum/{tag}: oos/ not found — {path}')
        continue
    d = {}
    for fname in os.listdir(path):
        if fname.endswith('_oos.pkl'):
            coin = fname.replace('_oos.pkl', '').upper()
            d[coin] = load_pkl(os.path.join(path, fname))
    momentum_oos[tag] = d

print('Stat arb pairs available:')
for p in statarb_oos:
    print(f'  {p.replace("_","/")}')

print('\nMomentum sleeves available (tag → coin):')
for tag, d in momentum_oos.items():
    for c in sorted(d):
        print(f'  {tag}:{c}')

---
## 2 · Sleeve Selection

- **`STATARB_SELECTION`** — list of pair labels, or `None` for all loaded pairs.
- **`MOMENTUM_SELECTION`** — `label → (strategy_tag, coin)`. The label is what appears in plots/tables — prefer `{COIN}_{strat}` so the same coin from two strategies is unambiguous.

In [ ]:
# ── stat arb sleeves ─────────────────────────────────────────────────────────
STATARB_SELECTION = None   # None = all loaded pairs  |  or e.g. ['ATOM_ARB','FIL_SNX']

# ── momentum sleeves ─────────────────────────────────────────────────────────
MOMENTUM_SELECTION = {
    # wf_testing_2 sleeves
    'BTC_wf2':  ('wf2', 'BTC'),
    'ETH_wf2':  ('wf2', 'ETH'),
    'SOL_wf2':  ('wf2', 'SOL'),
    'XRP_wf2':  ('wf2', 'XRP'),
    'AVAX_wf2': ('wf2', 'AVAX'),

    # bb_breakout_wf sleeves
    'BTC_bb':   ('bb',  'BTC'),
    'ETH_bb':   ('bb',  'ETH'),
    'AVAX_bb':  ('bb',  'AVAX'),
    'ADA_bb':   ('bb',  'ADA'),
    'NEAR_bb':  ('bb',  'NEAR'),
}

# ── resolve + flag anything missing ─────────────────────────────────────────
_sa_sel = STATARB_SELECTION if STATARB_SELECTION is not None else list(statarb_oos.keys())

sa_dfs, sa_missing = {}, []
for p in _sa_sel:
    if p in statarb_oos: sa_dfs[p] = statarb_oos[p]
    else:                sa_missing.append(p)

mom_dfs, mom_missing = {}, []
for label, (tag, coin) in MOMENTUM_SELECTION.items():
    if tag in momentum_oos and coin in momentum_oos[tag]:
        mom_dfs[label] = momentum_oos[tag][coin]
    else:
        mom_missing.append(f'{label} → {tag}:{coin}')

if sa_missing:  print(f'[stat arb missing] {sa_missing}')
if mom_missing: print(f'[momentum missing] {mom_missing}')

print(f'\nStat arb sleeves  ({len(sa_dfs)}):  {list(sa_dfs)}')
print(f'Momentum sleeves  ({len(mom_dfs)}): {list(mom_dfs)}')

print('\nSleeve diagnostic:')
for k, df in sa_dfs.items():
    print(f'  sa/{k:<10}  {sleeve_freq(df):<7}  {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
for s, df in mom_dfs.items():
    print(f'  mo/{s:<10}  {sleeve_freq(df):<7}  {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')

---
## 3 · Three-Level Weighting + Unified Sleeve Schema

Two things happen here:

**(a) Weights** — three levels, all auto-normalised.

| Level | Control | Options |
|---|---|---|
| 1 — across buckets | `STRATEGY_WEIGHTS` | `{'statarb': 0.3, 'momentum': 0.7}` |
| 2a — within stat arb | `STATARB_METHOD` | `'inverse_vol'` · `'equal'` · explicit `dict` |
| 2b — within momentum (across strategies) | `MOM_STRAT_WEIGHTS` | e.g. `{'bb': 0.6, 'wf2': 0.4}` · `None` = equal |
| 2b — within momentum (across coins) | `MOM_COIN_WEIGHTS` | e.g. `{'bb': {'BTC': 0.4, ...}}` · `None` = equal |

Final sleeve weight = `bucket × within_bucket × within_strategy`. All weight-building logic lives in `portfolio_metrics` (`build_momentum_weights`, `sa_inverse_vol_weights`) — the config cell below only sets the parameters.

**(b) Unified sleeve schema** — so `plot_portfolio_oos` can chart both buckets with one call, each sleeve is wrapped as a minimal DataFrame `{Close, position=1, position_size=1}` where `Close = (1 + bar_return).cumprod()`. Then `plot_portfolio_oos._strat_ret` just reads `Close.pct_change()` back out. Costs are baked in at this stage — never re-applied downstream.

> **Mixed-frequency note**: bb sleeves are hourly (~26 400 bars), wf2 and stat arb are daily (~1 000 bars). `plot_portfolio_oos` aligns them internally via `pd.concat(..., axis=1).fillna(0)` — NaN at non-overlapping timestamps becomes 0 (no position held that bar). The weight sweeps in §5 and §6 use the same alignment, so their numbers are directly comparable to the main chart.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LEVEL 1 — across strategy buckets
# ══════════════════════════════════════════════════════════════════════════════
STRATEGY_WEIGHTS = {'statarb': 0.30, 'momentum': 0.70}

# ══════════════════════════════════════════════════════════════════════════════
# LEVEL 2a — within stat arb   ('inverse_vol' | 'equal' | dict)
# ══════════════════════════════════════════════════════════════════════════════
STATARB_METHOD     = 'inverse_vol'
STATARB_VOL_METHOD = 'in_market'     # 'full' | 'in_market'
STATARB_VOL_WINDOW = None            # None = full OOS history | int = last N bars

# ══════════════════════════════════════════════════════════════════════════════
# LEVEL 2b — within momentum (strategy split × coin split)
# ══════════════════════════════════════════════════════════════════════════════
MOM_STRAT_WEIGHTS = {'bb': 0.4, 'wf2': 0.6}   # mirrors portfolio_master default
# MOM_STRAT_WEIGHTS = None   # ← uncomment for equal weight across strategies
MOM_COIN_WEIGHTS  = None    # e.g. {'bb': {'BTC': 0.4, ...}} |  None = equal within each strat

MOMENTUM_COST = 0.001   # per-leg cost applied to momentum bar returns
                        # stat arb pkls already carry baked net_returns (baseline cost=0.001)

# ── weight calculation ───────────────────────────────────────────────────────
if isinstance(STATARB_METHOD, dict):
    sa_w = norm_weights({k: STATARB_METHOD.get(k, 0) for k in sa_dfs})
elif STATARB_METHOD == 'equal':
    sa_w = {k: 1 / len(sa_dfs) for k in sa_dfs} if sa_dfs else {}
else:  # inverse_vol
    sa_w = sa_inverse_vol_weights(sa_dfs, STATARB_VOL_METHOD, STATARB_VOL_WINDOW)

mom_w          = build_momentum_weights(mom_dfs, MOMENTUM_SELECTION, MOM_STRAT_WEIGHTS, MOM_COIN_WEIGHTS)
sw             = norm_weights(STRATEGY_WEIGHTS)
sleeve_weights = build_sleeve_weights(sa_dfs, sa_w, mom_dfs, mom_w, STRATEGY_WEIGHTS)

# ── unified sleeve schema ───────────────────────────────────────────────────
sleeve_dfs = {}
for k, df in sa_dfs.items():
    sleeve_dfs[k] = wrap_as_sleeve(df['net_returns'].fillna(0))
for s, df in mom_dfs.items():
    sleeve_dfs[s] = wrap_as_sleeve(mom_bar_returns(df, MOMENTUM_COST))

# ── audit print ─────────────────────────────────────────────────────────────
print('═' * 70)
print('  LEVEL 1 — bucket allocation')
print('─' * 70)
for k, v in sw.items():
    print(f'  {k:<12}  {v*100:>6.2f}%')

print('\n' + '─' * 70)
print(f'  LEVEL 2a — STAT ARB sleeves  (method={STATARB_METHOD})')
print('─' * 70)
print(f'  {"Pair":<12}  {"within":>8}   {"bucket":>8}   {"final":>8}')
for k in sa_dfs:
    print(f'  {k.replace("_","/"):<12}  {sa_w[k]*100:>7.2f}%   {sw["statarb"]*100:>7.2f}%   {sleeve_weights[k]*100:>7.2f}%')

print('\n' + '─' * 70)
print('  LEVEL 2b — MOMENTUM sleeves')
print('─' * 70)
print(f'  {"Sleeve":<12}  {"within":>8}   {"bucket":>8}   {"final":>8}')
for s in mom_dfs:
    print(f'  {s:<12}  {mom_w[s]*100:>7.2f}%   {sw["momentum"]*100:>7.2f}%   {sleeve_weights[s]*100:>7.2f}%')
print('═' * 70)

---
## 4 · Combined Portfolio Performance

Main equity / drawdown / yearly-stats chart — rendered by `plot_portfolio_oos` exactly as in `portfolio_master`. Benchmark is BTC buy-and-hold (sourced from the BTC sleeve's raw `Close` — bypassing our unified wrapper so it's a real price series, not an equity curve). A cost stress test below sweeps 1× → 3× the base cost.

In [ ]:
SHOW_SLEEVES = list(sleeve_dfs.keys())   # or a manual subset

# BTC benchmark — grab raw Close from any available BTC sleeve
_btc_src = next((mom_dfs[s] for s, (t, c) in MOMENTUM_SELECTION.items() if c == 'BTC' and s in mom_dfs), None)
benchmark = {'BTC': _btc_src} if _btc_src is not None else None

metrics = plot_portfolio_oos(
    coin_dfs   = sleeve_dfs,
    weights    = sleeve_weights,
    show_coins = SHOW_SLEEVES,
    benchmark  = benchmark,
    show       = True,
    cost       = 0.0,   # bar returns already carry their own cost
    save_html  = None,
)

# ── cost stress test (re-wraps momentum sleeves at each cost level) ──────────
BASE_COST        = 0.001
cost_multipliers = (1.0, 1.5, 2.0, 3.0)

print(f"\n{'═'*72}")
print('PORTFOLIO TRANSACTION COST STRESS TEST  (momentum cost only)')
print(f"{'═'*72}")
print(f'{"Cost":>8} {"Mult":>6} {"Sharpe":>8} {"Return":>10} {"MaxDD":>10} {"Calmar":>8}')
print(f'{"─"*8} {"─"*6} {"─"*8} {"─"*10} {"─"*10} {"─"*8}')

for mult in cost_multipliers:
    c_mom = BASE_COST * mult
    sdfs = dict(sleeve_dfs)
    for s, df in mom_dfs.items():
        sdfs[s] = wrap_as_sleeve(mom_bar_returns(df, c_mom))
    with contextlib.redirect_stdout(io.StringIO()):
        m = plot_portfolio_oos(
            coin_dfs=sdfs, weights=sleeve_weights, show_coins=SHOW_SLEEVES,
            benchmark=benchmark, show=False, cost=0.0, save_html=None,
        )
    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0
    marker = ' <-- baseline' if mult == 1.0 else ''
    print(f'{c_mom:>8.4f} {mult:>5.1f}x {m["sharpe_ratio"]:>8.2f} '
          f'{m["total_return"]*100:>9.2f}% {m["max_drawdown"]*100:>9.2f}% {calmar:>8.2f}{marker}')

---
## 5 · Bucket Comparison · Correlation · Weight Sensitivity

Three extras from `combined_portfolio.py`, layered on top of `portfolio_master`'s template:

1. **Bucket comparison** — run `plot_portfolio_oos` headless on each bucket alone (using bucket-internal weights), then print StatArb / Momentum / Combined side-by-side.
2. **Correlation** — bucket-vs-bucket on the union index. Near zero = genuine diversification.
3. **Weight sweep** — top-level split from 0% → 100% stat arb in 5% steps, bucket internals held fixed.

In [ ]:
# ── bucket-only metrics (headless) ──────────────────────────────────────────
def _bucket_metrics(labels, internal_w):
    if not labels:
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        return plot_portfolio_oos(
            coin_dfs={k: sleeve_dfs[k] for k in labels},
            weights=internal_w, show_coins=labels,
            benchmark=benchmark, show=False, cost=0.0, save_html=None,
        )

m_sa  = _bucket_metrics(list(sa_dfs),  sa_w)
m_mom = _bucket_metrics(list(mom_dfs), mom_w)

def _fmt(m, key, fmt, scale=1, suffix=''):
    return f'{(m[key]*scale):>{fmt}}{suffix}' if m is not None else f'{"—":>10}'

print('═' * 62)
print(f'  {"":20}  {"StatArb":>10}  {"Momentum":>10}  {"Combined":>10}')
print('─' * 62)
for label, key, fmt, scale, sfx in [
    ('Total Return', 'total_return', '8.1f',  100, '%'),
    ('Sharpe Ratio', 'sharpe_ratio', '10.2f', 1,   '' ),
    ('Max Drawdown', 'max_drawdown', '8.1f',  100, '%'),
    ('Calmar Ratio', 'calmar_ratio', '10.2f', 1,   '' ),
]:
    a = _fmt(m_sa,    key, fmt, scale, sfx)
    b = _fmt(m_mom,   key, fmt, scale, sfx)
    c = _fmt(metrics, key, fmt, scale, sfx)
    print(f'  {label:<20}  {a:>10}  {b:>10}  {c:>10}')
print('═' * 62)

# ── bucket equity curves for correlation + sweep ─────────────────────────────
sa_eq = m_sa['equity_curve']  if m_sa  else pd.Series(dtype=float)
mo_eq = m_mom['equity_curve'] if m_mom else pd.Series(dtype=float)
_idx  = sa_eq.index.union(mo_eq.index)
sa_r  = sa_eq.pct_change().fillna(0).reindex(_idx).fillna(0)
mo_r  = mo_eq.pct_change().fillna(0).reindex(_idx).fillna(0)

print(f'\n  Stat Arb vs Momentum correlation: {sa_r.corr(mo_r):.4f}')

# ── top-level weight sweep ───────────────────────────────────────────────────
_sweep = sweep_top_level(sa_eq, mo_eq, step=5, current_sa_weight=sw['statarb'])

print('\n' + '═' * 62)
print('  WEIGHT SENSITIVITY — top-level split')
print('─' * 62)
print(f'  {"SA%":>5}  {"MOM%":>5}  {"Sharpe":>7}  {"Return":>8}  {"MaxDD":>7}  {"Calmar":>7}')
print(f'  {"-"*5}  {"-"*5}  {"-"*7}  {"-"*8}  {"-"*7}  {"-"*7}')
for r in _sweep:
    marker = ' <-- current' if r['is_current'] else ''
    print(f'  {r["sa_pct"]:>5}  {r["mom_pct"]:>5}  {r["sharpe_ratio"]:>7.2f}  '
          f'{r["total_return"]*100:>7.1f}%  {r["max_drawdown"]*100:>7.1f}%  '
          f'{r["calmar_ratio"]:>7.2f}{marker}')
print('═' * 62)

---
## 6 · Momentum-Internal Optimisation

Two analyses to answer: *what is the optimal portfolio within momentum, and across coins?*

1. **Strategy split sweep** — bb vs wf2 weight from 0% to 100% in 5% steps (coin weights held fixed per the config above). Identifies the optimal blend of the two momentum strategies.
2. **Per-coin contribution** — each sleeve run in isolation, sorted by Sharpe. Use this to inform `MOM_COIN_WEIGHTS` in the config cell — tilt up the stronger coins, tilt down the weaker ones.

In [ ]:
# ── momentum strategy weight sweep (bb vs wf2) ───────────────────────────────
_strat_grid = [
    {'bb': bb_pct / 100, 'wf2': (100 - bb_pct) / 100}
    for bb_pct in range(0, 101, 5)
]

_mom_sweep = sweep_momentum_strategy(
    mom_dfs            = mom_dfs,
    mom_selection      = MOMENTUM_SELECTION,
    strat_weights_grid = _strat_grid,
    coin_weights       = MOM_COIN_WEIGHTS,
    cost               = MOMENTUM_COST,
)

_cur_bb      = norm_weights(MOM_STRAT_WEIGHTS or {'bb': 1, 'wf2': 1}).get('bb', 0)
_best_sharpe = max(_mom_sweep, key=lambda r: r['sharpe_ratio'])
_best_calmar = max(_mom_sweep, key=lambda r: r['calmar_ratio'])

print('═' * 72)
print('  MOMENTUM STRATEGY WEIGHT SWEEP  (bb vs wf2, coin weights fixed)')
print('─' * 72)
print(f'  {"bb%":>5}  {"wf2%":>5}  {"Sharpe":>7}  {"Return":>9}  {"MaxDD":>7}  {"Calmar":>7}')
print(f'  {"-"*5}  {"-"*5}  {"-"*7}  {"-"*9}  {"-"*7}  {"-"*7}')
for r in _mom_sweep:
    bb_pct  = round(r['strat_weights']['bb']  * 100)
    wf2_pct = round(r['strat_weights']['wf2'] * 100)
    tags = []
    if abs(r['strat_weights']['bb'] - _cur_bb) < 1e-9:
        tags.append('current')
    if r is _best_sharpe:
        tags.append('best Sharpe')
    if r is _best_calmar and r is not _best_sharpe:
        tags.append('best Calmar')
    suffix = f'  <-- {", ".join(tags)}' if tags else ''
    print(f'  {bb_pct:>5}  {wf2_pct:>5}  {r["sharpe_ratio"]:>7.2f}  '
          f'{r["total_return"]*100:>8.1f}%  {r["max_drawdown"]*100:>7.1f}%  '
          f'{r["calmar_ratio"]:>7.2f}{suffix}')
print('═' * 72)

In [ ]:
# ── per-coin contribution (each momentum sleeve run in isolation) ─────────────
print('═' * 72)
print('  MOMENTUM PER-COIN ANALYSIS  (each sleeve run in isolation)')
print('─' * 72)
print(f'  {"Sleeve":<12}  {"Strategy":>8}  {"Sharpe":>7}  {"Return":>9}  {"MaxDD":>8}  {"Calmar":>7}')
print(f'  {"-"*12}  {"-"*8}  {"-"*7}  {"-"*9}  {"-"*8}  {"-"*7}')

_coin_rows = []
for s, df in mom_dfs.items():
    tag, coin = MOMENTUM_SELECTION[s]
    _bar_r = mom_bar_returns(df, MOMENTUM_COST)
    _df_bt = pd.DataFrame({'strategy_returns': _bar_r, 'position': 1}, index=_bar_r.index)
    with contextlib.redirect_stdout(io.StringIO()):
        _m = backtest(_df_bt, cost=0.0, show_plot=False)
    _calmar = _m['total_return'] / abs(_m['max_drawdown']) if _m['max_drawdown'] != 0 else 0.0
    _coin_rows.append({
        'sleeve': s, 'tag': tag,
        'sharpe': _m['sharpe_ratio'], 'ret': _m['total_return'],
        'dd': _m['max_drawdown'],     'calmar': _calmar,
    })

_coin_rows.sort(key=lambda r: r['sharpe'], reverse=True)
for r in _coin_rows:
    print(f'  {r["sleeve"]:<12}  {r["tag"]:>8}  {r["sharpe"]:>7.2f}  '
          f'{r["ret"]*100:>8.1f}%  {r["dd"]*100:>8.1f}%  {r["calmar"]:>7.2f}')

print('═' * 72)
print('\n  To tilt towards stronger coins set MOM_COIN_WEIGHTS in cell 7, e.g.:')
print("  MOM_COIN_WEIGHTS = {'bb': {'BTC': 0.4, 'ETH': 0.3, 'AVAX': 0.15, 'ADA': 0.1, 'NEAR': 0.05},")
print("                       'wf2': {'BTC': 0.35, 'ETH': 0.25, 'XRP': 0.2, 'SOL': 0.1, 'AVAX': 0.1}}")

---
## 7 · Closed-Trade Equity (realized P&L only)

Steps only on sleeve exits — each step is the weighted contribution of one closed trade. Uses the **original** per-sleeve dfs (real `position` column) for entry/exit detection. Hides intra-trade MTM — read alongside the main chart above.

In [ ]:
# sa_dfs have 'net_returns' (pre-baked, cost already applied)
# mom_dfs have Close/position → cost applied here
plot_closed_trade_equity(
    {**sa_dfs, **mom_dfs},
    weights = sleeve_weights,
    cost    = MOMENTUM_COST,
)